# Human Action Classification — Transformer Deployment (ngrok)

Mirrors `Human_Action_Classification_deployment_with_ngrok.ipynb` but
uses the Transformer model instead of the LSTM.

**Pre-requisite:** Run `transformer_train.ipynb` first so that
`models/saved_model.ckpt` exists.


## 1 · Install dependencies

In [ ]:
import torch
assert torch.__version__.startswith('1.8'), 'Need torch 1.8 for detectron2'
!pip install detectron2 -f \
    https://dl.fbaipublicfiles.com/detectron2/wheels/cu101/torch1.8/index.html --quiet
!pip install pytorch-lightning einops flask --quiet

## 2 · Clone repo

In [ ]:
!git clone https://github.com/shsarv/Huam-Activity-Detection
%cd Huam-Activity-Detection/
!pip install -r requirements.txt --quiet

## 3 · Place transformer files

Upload (or copy) the following two files into the repo:

| File | Destination |
|------|-------------|
| `src/transformer.py` | `src/transformer.py` |
| `app.py` (updated) | `app.py` |
| `models/saved_model.ckpt` | `models/saved_model.ckpt` |

If running in Colab, use the cell below to upload them.

In [ ]:
from google.colab import files
import os, shutil

print('Upload src/transformer.py, updated app.py, and models/saved_model.ckpt')
uploaded = files.upload()

os.makedirs('src', exist_ok=True)
os.makedirs('models', exist_ok=True)

for name in uploaded:
    if name == 'transformer.py':
        shutil.move(name, 'src/transformer.py')
    elif name == 'app.py':
        shutil.move(name, 'app.py')
    elif name == 'saved_model.ckpt':
        shutil.move(name, 'models/saved_model.ckpt')
    print(f'  placed: {name}')

## 4 · Verify model loads correctly

In [ ]:
from src.transformer import ActionClassificationTransformer

model = ActionClassificationTransformer.load_from_checkpoint('models/saved_model.ckpt')
model.eval()
print('✅ Transformer checkpoint loaded successfully')
print(model.hparams)

## 5 · Download ngrok

In [ ]:
!if [ ! -f ./ngrok ]; then \
  wget -q https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip; \
  unzip -q -o ngrok-stable-linux-amd64.zip; \
fi
print('ngrok ready')

## 6 · Start Flask + ngrok

In [ ]:
port = 5000

get_ipython().system_raw(f'python app.py &')
get_ipython().system_raw(f'./ngrok http {port} &')

## 7 · Get public URL

In [ ]:
import time, json, urllib.request

time.sleep(2)  # wait for ngrok to start
data = json.load(urllib.request.urlopen('http://localhost:4040/api/tunnels'))
url  = data['tunnels'][0]['public_url']
print(f'🌐 Public URL: {url}')